[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_10/notebook_10_1_shap_feature_importance.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.1: Feature Importance and SHAP Values

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: Making Black Box Models Transparent

A cardiologist reviews an AI's prediction:

```
Patient: 58-year-old male
AI Prediction: HIGH RISK of heart attack (0.83 probability)
Recommendation: Immediate catheterization
```

**The critical question**: *Why* does the AI think this patient is high risk?

Without an answer, the cardiologist faces an impossible choice:
- Trust blindly → risk unnecessary invasive procedure
- Ignore completely → risk missing life-threatening MI

**This notebook teaches you how to answer "why"** using:
1. **Permutation importance**: Which features matter most globally?
2. **SHAP values**: How does each feature contribute to this specific prediction?
3. **Clinical interpretation**: Translating math into actionable insights

## Learning Objectives

By the end of this notebook, you will:
- Understand the difference between global and local explanations
- Implement permutation importance for model validation
- Compute SHAP values for individual predictions
- Visualize feature importance with clinical annotations
- Detect spurious correlations that accuracy metrics miss
- Communicate AI predictions to clinicians effectively

---

## Part 1: Setup and Data

We'll use a cardiovascular disease prediction dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook demonstrates how to explain AI predictions using SHAP.")
print("We'll make black box models transparent and clinically trustworthy.\n")

In [ ]:
def create_cvd_dataset(n_samples=5000, include_spurious=False):
    """
    Create cardiovascular disease prediction dataset.

    Features (clinically relevant):
    - Age, gender, BMI
    - Blood pressure (systolic, diastolic)
    - Cholesterol (total, LDL, HDL)
    - Blood glucose
    - Smoking status, family history

    If include_spurious=True, adds:
    - Hospital ID (spurious correlation for demonstration)
    """
    data = []

    for i in range(n_samples):
        # Demographics
        age = np.random.normal(60, 15)
        age = np.clip(age, 30, 90)
        gender = np.random.choice(['M', 'F'])

        # Calculate CVD risk based on established clinical factors
        base_risk = 0.15

        # Age effect (exponential increase)
        if age > 65:
            base_risk *= 1.8
        elif age > 50:
            base_risk *= 1.3

        # Gender effect (males higher risk at younger ages)
        if gender == 'M' and age < 65:
            base_risk *= 1.4

        has_cvd = np.random.random() < base_risk

        # Clinical features (values depend on CVD status)
        if has_cvd:
            systolic_bp = np.random.normal(150, 20)
            diastolic_bp = np.random.normal(95, 12)
            total_chol = np.random.normal(250, 40)
            ldl_chol = np.random.normal(160, 35)
            hdl_chol = np.random.normal(40, 10)
            triglycerides = np.random.normal(180, 50)
            blood_glucose = np.random.normal(120, 30)
            bmi = np.random.normal(30, 5)
            smoking = np.random.random() < 0.40
            family_history = np.random.random() < 0.60
            exercise_hours = np.random.normal(1, 1)
        else:
            systolic_bp = np.random.normal(125, 15)
            diastolic_bp = np.random.normal(80, 10)
            total_chol = np.random.normal(200, 30)
            ldl_chol = np.random.normal(110, 25)
            hdl_chol = np.random.normal(55, 12)
            triglycerides = np.random.normal(130, 40)
            blood_glucose = np.random.normal(95, 20)
            bmi = np.random.normal(25, 4)
            smoking = np.random.random() < 0.15
            family_history = np.random.random() < 0.30
            exercise_hours = np.random.normal(3, 1.5)

        # Spurious feature (for demonstration of data leakage detection)
        if include_spurious:
            # Hospital A sees sicker patients (selection bias)
            if has_cvd:
                hospital_id = np.random.choice(['A', 'B', 'C'], p=[0.6, 0.2, 0.2])
            else:
                hospital_id = np.random.choice(['A', 'B', 'C'], p=[0.2, 0.4, 0.4])
        else:
            hospital_id = np.random.choice(['A', 'B', 'C'])

        data.append({
            'age': age,
            'gender': gender,
            'systolic_bp': systolic_bp,
            'diastolic_bp': diastolic_bp,
            'total_cholesterol': total_chol,
            'ldl_cholesterol': ldl_chol,
            'hdl_cholesterol': hdl_chol,
            'triglycerides': triglycerides,
            'blood_glucose': blood_glucose,
            'bmi': bmi,
            'smoking': int(smoking),
            'family_history': int(family_history),
            'exercise_hours_per_week': exercise_hours,
            'hospital_id': hospital_id,
            'cvd': int(has_cvd)
        })

    df = pd.DataFrame(data)

    # Clip to reasonable ranges
    df['age'] = df['age'].clip(30, 90)
    df['systolic_bp'] = df['systolic_bp'].clip(90, 200)
    df['diastolic_bp'] = df['diastolic_bp'].clip(50, 120)
    df['total_cholesterol'] = df['total_cholesterol'].clip(120, 400)
    df['ldl_cholesterol'] = df['ldl_cholesterol'].clip(50, 300)
    df['hdl_cholesterol'] = df['hdl_cholesterol'].clip(20, 100)
    df['triglycerides'] = df['triglycerides'].clip(50, 400)
    df['blood_glucose'] = df['blood_glucose'].clip(60, 250)
    df['bmi'] = df['bmi'].clip(15, 50)
    df['exercise_hours_per_week'] = df['exercise_hours_per_week'].clip(0, 10)

    return df

# Create dataset
df = create_cvd_dataset(n_samples=5000, include_spurious=False)

print("Dataset created!")
print(f"\nTotal samples: {len(df):,}")
print(f"CVD prevalence: {df['cvd'].mean():.1%}")
print("\nFirst few rows:")
df.head(10)

In [ ]:
# Prepare data
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=['gender', 'hospital_id'], drop_first=False)

feature_cols = [col for col in df_encoded.columns if col != 'cvd']
X = df_encoded[feature_cols].values
y = df_encoded['cvd'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nFeatures: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")

## Part 2: Train Model (Black Box)

We'll train a Random Forest—accurate but opaque.

In [ ]:
# Train Random Forest
print("Training Random Forest model...\n")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("✓ Model trained successfully")
print(f"\nTest Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No CVD', 'CVD']))

print("\n📦 Black Box Problem:")
print("The model performs well, but we don't know WHY it makes predictions.")
print("Let's open the black box using explainability methods...\n")

## Part 3: Permutation Importance (Global Explanation)

**Question**: Which features matter most across all predictions?

**Method**: Randomly shuffle each feature's values and measure performance drop.

In [ ]:
print("Computing permutation importance...\n")
print("(This may take 30-60 seconds)\n")

# Compute permutation importance
perm_importance = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring='roc_auc'
)

# Create DataFrame for easier analysis
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("✓ Permutation importance computed\n")
print("Top 10 Most Important Features:\n")
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Visualize permutation importance
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 15
top_features = importance_df.head(top_n)

colors = ['darkred' if imp > 0.01 else 'orange' if imp > 0.005 else 'gray'
          for imp in top_features['importance']]

bars = ax.barh(range(len(top_features)), top_features['importance'],
               xerr=top_features['std'], color=colors, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'], fontsize=11)
ax.set_xlabel('Importance (Drop in AUC when permuted)', fontsize=12, fontweight='bold')
ax.set_title('Permutation Feature Importance\n(Higher = More Important)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, row) in enumerate(zip(bars, top_features.itertuples())):
    ax.text(row.importance + 0.001, i, f'{row.importance:.4f}',
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 CLINICAL INTERPRETATION:")
print("\nTop 5 drivers of CVD prediction:")
for idx, row in importance_df.head(5).iterrows():
    feature = row['feature']
    importance = row['importance']

    # Clinical context
    if 'age' in feature.lower():
        context = "✓ Age is a known CVD risk factor"
    elif 'chol' in feature.lower():
        context = "✓ Cholesterol levels predict CVD"
    elif 'bp' in feature.lower() or 'blood_pressure' in feature.lower():
        context = "✓ Blood pressure is major CVD risk factor"
    elif 'glucose' in feature.lower():
        context = "✓ Glucose (diabetes marker) increases CVD risk"
    elif 'bmi' in feature.lower():
        context = "✓ Obesity is CVD risk factor"
    elif 'smoking' in feature.lower():
        context = "✓ Smoking is major modifiable risk factor"
    elif 'hospital' in feature.lower():
        context = "⚠️ SPURIOUS: Hospital ID shouldn't matter (data leakage?)"
    else:
        context = ""

    print(f"  {idx+1}. {feature}: {importance:.4f}  {context}")

print("\n✅ The model appropriately prioritizes clinical risk factors.")
print("   This builds trust in the model's reasoning.\n")

## Part 4: SHAP Values (Local + Global Explanation)

SHAP (SHapley Additive exPlanations) provides:
- **Global**: Feature importance across all predictions
- **Local**: Contribution of each feature to a specific prediction

### Why SHAP?

1. **Theoretically grounded**: Based on game theory (Shapley values)
2. **Additive**: SHAP values sum to the difference between prediction and baseline
3. **Consistent**: If a feature becomes more important, its SHAP value doesn't decrease
4. **Local accuracy**: Faithfully represents model's reasoning for each prediction

In [ ]:
# Initialize SHAP explainer
print("Initializing SHAP explainer...\n")
print("(TreeExplainer is fast for tree-based models like Random Forest)\n")

explainer = shap.TreeExplainer(model)

# Compute SHAP values for test set
print("Computing SHAP values for test set...")
print("(This may take 1-2 minutes for 1000 samples)\n")

shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is a list [class_0, class_1]
# We want class 1 (CVD positive)
if isinstance(shap_values, list):
    shap_values_cvd = shap_values[1]
else:
    shap_values_cvd = shap_values

print("✓ SHAP values computed successfully\n")
print(f"Shape: {shap_values_cvd.shape}")
print(f"(n_samples={shap_values_cvd.shape[0]}, n_features={shap_values_cvd.shape[1]})")

### SHAP Summary Plot (Global Feature Importance)

In [ ]:
# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_cvd, X_test, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot: Feature Importance Distribution',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 How to Read This Plot:")
print("\n  - Each dot is one patient")
print("  - Y-axis: Features (top = most important)")
print("  - X-axis: SHAP value (impact on prediction)")
print("    • Positive (right) = increases CVD risk")
print("    • Negative (left) = decreases CVD risk")
print("  - Color: Feature value (red = high, blue = low)")
print("\n  Example: Age")
print("    • Red dots (older patients) are on the right → higher age increases risk")
print("    • Blue dots (younger patients) are on the left → lower age decreases risk")
print("    • This makes clinical sense! ✓\n")

### SHAP Bar Plot (Average Absolute Impact)

In [ ]:
# Bar plot (mean absolute SHAP values)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_cvd, X_test, feature_names=feature_cols,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP value|)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Global Feature Importance (SHAP):")
print("\nThis shows the average magnitude of each feature's impact.")
print("Higher bars = more important for predictions overall.\n")

## Part 5: Local Explanations - Individual Patient

Now the real clinical value: explaining specific predictions.

In [ ]:
# Select a high-risk patient
high_risk_idx = np.where(y_prob > 0.75)[0]

if len(high_risk_idx) > 0:
    patient_idx = high_risk_idx[0]
else:
    patient_idx = np.argmax(y_prob)

patient_features = X_test[patient_idx]
patient_prob = y_prob[patient_idx]
patient_true_label = y_test[patient_idx]

print("="*80)
print("HIGH-RISK PATIENT ANALYSIS")
print("="*80 + "\n")

print(f"Patient ID: {patient_idx}")
print(f"AI Prediction: {'HIGH RISK' if patient_prob > 0.5 else 'LOW RISK'} ({patient_prob:.1%} probability)")
print(f"True Label: {'CVD' if patient_true_label == 1 else 'No CVD'}")
print(f"\nPatient Features:")

for feat_name, feat_val in zip(feature_cols, patient_features):
    # Format feature values nicely
    if feat_val == 0 or feat_val == 1:
        # Binary feature
        display_val = 'Yes' if feat_val == 1 else 'No'
    else:
        # Continuous feature
        display_val = f"{feat_val:.1f}"

    print(f"  {feat_name}: {display_val}")

print("\n" + "="*80)

### SHAP Waterfall Plot (Step-by-Step Explanation)

In [ ]:
# Check if expected_value is an array/list (multi-class) or scalar (single output)
base_val = explainer.expected_value
if hasattr(base_val, '__len__') and len(base_val) > 1:
    # If it has multiple classes, grab the one for CVD Risk (Index 1)
    base_val = base_val[1]

# Ensure it is a pure Python float (removes numpy array wrappers)
base_val = float(base_val)

# PREPARE EXPLANATION OBJECT
explanation = shap.Explanation(
    values=shap_values_cvd[patient_idx][:, 1], # Contribution to Class 1
    base_values=base_val,                      # Scalar baseline risk
    data=X_test[patient_idx],                  # Feature values
    feature_names=feature_cols
)

# 3. PLOT
plt.figure(figsize=(10, 6))
shap.plots.waterfall(explanation, show=False)

plt.title(f'SHAP Waterfall: Patient {patient_idx} ({patient_prob:.1%} CVD Risk)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### SHAP Force Plot (Alternative Visualization)

In [ ]:
# 1. Robustly extract scalar base value for Class 1
base_val = explainer.expected_value
if hasattr(base_val, '__len__') and len(base_val) > 1:
    base_val = base_val[1] # Select Class 1 base value
base_val = float(base_val) # Ensure it's a float

# 2. Select SHAP values for Class 1 only
# Input shape was likely (features, 2) -> we want (features,)
shap_values_single = shap_values_cvd[patient_idx][:, 1]

plt.figure(figsize=(14, 3))

# 3. Generate Plot
shap.force_plot(
    base_value=base_val,
    shap_values=shap_values_single,
    features=X_test[patient_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=False
)

plt.title(f'SHAP Force Plot: Patient {patient_idx}', fontsize=14, fontweight='bold', y=1.5)
plt.tight_layout()
plt.savefig(f'shap_force_patient_{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Force Plot Interpretation:")
print(f"  - Base value: {base_val:.3f} (average model output)")
print("  - Red bars: Features pushing the prediction HIGHER")
print("  - Blue bars: Features pushing the prediction LOWER")
print(f"  - Final Output: {base_val + shap_values_single.sum():.3f}\n")

### Clinical Interpretation (Plain Language)

In [ ]:
import numpy as np

# 1. FIX: Select only Class 1 (CVD Risk) contributions
# This turns the shape from (n_features, 2) to (n_features,)
patient_shap = shap_values_cvd[patient_idx][:, 1]

# 2. FIX: Robust baseline extraction (as discussed previously)
baseline = explainer.expected_value
if hasattr(baseline, '__len__') and len(baseline) > 1:
    baseline = baseline[1]
baseline = float(baseline)

# 3. Ensure we have the raw feature values for display
patient_features = X_test[patient_idx]

# Get top contributing features
# Now shap_val is a simple float, so sorting works!
shap_importance = [(feat_name, shap_val) for feat_name, shap_val in zip(feature_cols, patient_shap)]
shap_importance_sorted = sorted(shap_importance, key=lambda x: abs(x[1]), reverse=True)

print("\n" + "="*80)
print("CLINICAL EXPLANATION FOR PHYSICIAN")
print("="*80 + "\n")

print(f"Patient {patient_idx}: {'HIGH RISK' if patient_prob > 0.5 else 'LOW RISK'} of CVD ({patient_prob:.1%})\n")

# Note: If your SHAP values are Log-Odds (default), this baseline percentage might look small/odd
# unless you converted explainer output to probability space.
print(f"Baseline risk (average patient): {baseline:.1%}\n")

print("Top 5 factors INCREASING risk:\n")
positive_factors = [(f, s) for f, s in shap_importance_sorted if s > 0]

for i, (feat, shap_val) in enumerate(positive_factors[:5], 1):
    # Find original feature value
    feat_idx = feature_cols.index(feat)
    feat_val = patient_features[feat_idx]

    # Format value
    if feat_val == 0 or feat_val == 1:
        val_str = 'Yes' if feat_val == 1 else 'No'
    else:
        val_str = f"{feat_val:.1f}"

    # Clinical context logic
    if 'age' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}" # Changed % to raw score for safety
        note = "(Age is non-modifiable risk factor)"
    elif 'chol' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Consider statin therapy if elevated)"
    elif 'bp' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Consider antihypertensive if >140/90)"
    elif 'smoking' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Smoking cessation counseling recommended)"
    else:
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = ""

    print(f"  {i}. {feat} = {val_str}")
    print(f"     {risk_change}  {note}")
    print()

print("Top 3 factors DECREASING risk:\n")
negative_factors = [(f, s) for f, s in shap_importance_sorted if s < 0]

for i, (feat, shap_val) in enumerate(negative_factors[:3], 1):
    feat_idx = feature_cols.index(feat)
    feat_val = patient_features[feat_idx]

    if feat_val == 0 or feat_val == 1:
        val_str = 'Yes' if feat_val == 1 else 'No'
    else:
        val_str = f"{feat_val:.1f}"

    print(f"  {i}. {feat} = {val_str}")
    print(f"     Decreases risk score by {abs(shap_val):.3f}")
    print()

print("="*80)
print("\n💡 CLINICAL DECISION SUPPORT:")
print("\nThis explanation helps the clinician:")
print("  ✓ Verify the AI's reasoning aligns with clinical knowledge")
print("  ✓ Identify modifiable risk factors for intervention")
print("  ✓ Communicate risk to the patient in understandable terms")
print("  ✓ Decide whether to order additional tests or treatments\n")

## Part 6: Detecting Spurious Correlations

Let's see how explainability catches data leakage.

In [ ]:
# Create dataset WITH spurious correlation
print("Creating new dataset with spurious correlation (hospital ID)...\n")

df_spurious = create_cvd_dataset(n_samples=5000, include_spurious=True)

# Prepare data
df_spurious_encoded = pd.get_dummies(df_spurious, columns=['gender', 'hospital_id'], drop_first=False)
feature_cols_spurious = [col for col in df_spurious_encoded.columns if col != 'cvd']
X_spurious = df_spurious_encoded[feature_cols_spurious].values
y_spurious = df_spurious_encoded['cvd'].values

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_spurious, y_spurious, test_size=0.2, random_state=42, stratify=y_spurious
)

# Train model
model_spurious = RandomForestClassifier(
    n_estimators=100, max_depth=12, min_samples_split=20, random_state=42, n_jobs=-1
)
model_spurious.fit(X_train_sp, y_train_sp)

y_prob_sp = model_spurious.predict_proba(X_test_sp)[:, 1]

print(f"Model with spurious feature:")
print(f"  AUC: {roc_auc_score(y_test_sp, y_prob_sp):.3f}")
print("\n⚠️  High accuracy doesn't guarantee the model is learning the right things!\n")

In [ ]:
# Compute SHAP for spurious model
print("Computing SHAP for spurious model...\n")

explainer_sp = shap.TreeExplainer(model_spurious)
shap_values_sp = explainer_sp.shap_values(X_test_sp)

if isinstance(shap_values_sp, list):
    shap_values_sp_cvd = shap_values_sp[1]
else:
    shap_values_sp_cvd = shap_values_sp

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_sp_cvd, X_test_sp, feature_names=feature_cols_spurious,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance (WITH Spurious Feature)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_spurious_model.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  ALERT: Data Leakage Detected!\n")
print("Notice that 'hospital_id' appears as a top feature.")
print("\nWhy this is problematic:")
print("  - Hospital ID has no causal relationship with CVD")
print("  - It's a proxy for patient selection bias (Hospital A sees sicker patients)")
print("  - Model will fail when deployed to new hospitals")
print("  - Model learns 'where patient was seen' instead of 'clinical risk factors'")
print("\n✅ Explainability caught this issue!")
print("   Without SHAP, we might deploy a flawed model.\n")

## Summary and Key Takeaways

### What We Learned

1. **Permutation Importance (Global)**:
   - Identifies which features matter most across all predictions
   - Fast and model-agnostic
   - Useful for quick model validation

2. **SHAP Values (Global + Local)**:
   - Global: Summary plot shows feature importance and effect directions
   - Local: Waterfall/force plots explain individual predictions
   - Theoretically grounded (Shapley values from game theory)
   - Additive: SHAP values sum to prediction

3. **Clinical Applications**:
   - **Build trust**: Physicians can verify AI reasoning aligns with medical knowledge
   - **Detect errors**: Catch spurious correlations and data leakage
   - **Guide decisions**: Identify modifiable risk factors for intervention
   - **Communicate**: Explain predictions to patients in plain language

4. **Practical Workflow**:
   ```
   1. Train model → evaluate accuracy
   2. Compute permutation importance → check top features make clinical sense
   3. Compute SHAP values → generate global and local explanations
   4. Review explanations with clinicians → iterate if needed
   5. Deploy with explanation interface for real-time clinical use
   ```

### Best Practices

✅ **Always explain before deploying**: Don't deploy black boxes in healthcare  
✅ **Validate explanations**: Check that top features align with medical knowledge  
✅ **Use multiple methods**: SHAP + permutation importance provide complementary insights  
✅ **Make it actionable**: Focus explanations on modifiable risk factors  
✅ **Test for spurious correlations**: Use explainability to catch data leakage  
✅ **Iterate with clinicians**: User testing ensures explanations are understandable  

### Limitations

⚠️ **Association ≠ Causation**: High SHAP value doesn't mean changing the feature will change the outcome  
⚠️ **Computational cost**: SHAP can be slow for very large datasets (use TreeSHAP for tree models)  
⚠️ **Correlated features**: SHAP may distribute importance unpredictably between correlated features  
⚠️ **Post-hoc explanations**: Methods like SHAP explain what the model *did*, not necessarily what it *should* do  

### Next Steps

In the next notebooks, we'll explore:
- **Notebook 10.2**: LIME for local explanations (alternative to SHAP)
- **Notebook 10.3**: GradCAM for medical imaging (explaining CNNs)
- **Notebook 10.4**: Counterfactual explanations ("what if" scenarios)

---

**Remember**: In healthcare AI, explainability is not a luxury—it's a necessity for safety, trust, and clinical adoption.

*"A model you don't understand is a model you shouldn't trust with patient care."*